# Machine Learning with Sklearn

This notebook covers two key machine learning tasks:

**Part 1 — KNN Classification** using the Breast Cancer dataset  
**Part 2 — Linear Regression** using the California Housing dataset

Both follow the standard ML pipeline:
1. Load the dataset
2. Split into train and test sets
3. Train the model
4. Predict on unseen data
5. Evaluate performance

---
# Part 1: KNN Classification — Breast Cancer Dataset

We use the **Breast Cancer dataset** from sklearn to classify tumours as:
- **0** = Malignant (cancerous)
- **1** = Benign (non-cancerous)

K-Nearest Neighbours (KNN) classifies a new data point by looking at the `k` closest training points and taking a majority vote.

## Step 1 — Import Libraries

In [25]:
import numpy as np
import sklearn
from sklearn import datasets
import matplotlib.pyplot as plt

## Step 2 — Load the Dataset

- `bb.data` → the 30 feature measurements per tumour sample (e.g. size, texture, smoothness)
- `bb.target` → the class labels (0 = malignant, 1 = benign)

We assign them to `X` (features) and `y` (labels) — standard ML convention where:
- `X` is a **matrix** of features (capital letter because multiple columns)
- `y` is a **vector** of labels (lowercase because single column)

In [26]:
# Load the Breast Cancer dataset (predefined by sklearn)
bb = datasets.load_breast_cancer()

# X = feature matrix (30 measurements per tumour sample)
X = bb.data

# y = target labels (0 = malignant, 1 = benign)
y = bb.target

## Step 3 — Split into Train and Test Sets

We split **before** fitting to avoid **data leakage** (the model must never see test data during training).

- `test_size=0.2` → 80% training, 20% testing
- The correct variable order is always: `X_train, X_test, y_train, y_test`

| Variable | Role |
|---|---|
| `Xtrain, ytrain` | Used to **teach** the model |
| `Xtest` | **Questions** fed into the model |
| `ytest` | **Answer key** used only for evaluation |

In [27]:
from sklearn.model_selection import train_test_split

# Split: 80% training data, 20% test data
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2)

## Step 4 — Train the KNN Model and Predict

- `n_neighbors=5` → look at the **5 nearest neighbours** to make a decision (default, common starting point)
- `.fit(Xtrain, ytrain)` → the model **learns** from training features + labels
- `.predict(Xtest)` → the model **predicts** labels for unseen test features

> Note: we only pass `Xtest` (not `ytest`) to predict — passing the real answers would be cheating!

In [28]:
from sklearn.neighbors import KNeighborsClassifier

# Create the KNN model with k=5 neighbours
knn = KNeighborsClassifier(n_neighbors=5)

# Train the model using only training data (never test data)
knn.fit(Xtrain, ytrain)

# Predict labels for the unseen test features
ypredtest = knn.predict(Xtest)

## Step 5 — Confusion Matrix

A **confusion matrix** shows correct vs incorrect predictions for each class.

For binary classification (2 classes):

|  | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | True Negative (TN) | False Positive (FP) |
| **Actual 1** | False Negative (FN) | True Positive (TP) |

- **Diagonal values** = correct predictions
- **Off-diagonal values** = mistakes
- Order of arguments: `confusion_matrix(actual, predicted)` — **actual always first**

In [29]:
from sklearn.metrics import confusion_matrix

# Build confusion matrix — actual labels first, predicted second
cm = confusion_matrix(ytest, ypredtest)

# Visualise as a colour map (brighter = higher value)
plt.matshow(cm)

# Print the raw numbers to see exact counts
print(cm)

## Step 6 — Custom Accuracy Function

**Accuracy** = proportion of correct predictions out of all predictions.

$$Accuracy = \frac{\text{Total Correct Predictions}}{\text{Total Predictions}}$$

Using the confusion matrix:
- `np.diag(cm)` → extracts the diagonal (correct predictions per class)
- `.sum()` → adds them all up
- `cm.sum()` → total number of samples

In [39]:
def accuracy(ytest, ypredtest):
    # Build confusion matrix
    cm = confusion_matrix(ytest, ypredtest)
    
    # Diagonal = correct predictions, cm.sum() = total samples
    accuracy = np.diag(cm).sum() / cm.sum()
    
    return accuracy

## Step 7 — Custom Precision Function

**Precision** = of all the times the model predicted a class, how often was it correct?

$$Precision_i = \frac{TP}{TP + FP}$$

Precision is calculated **per column** of the confusion matrix:
- `cm.T` → transpose the matrix so columns become rows (easier to loop over)
- `col[i]` → diagonal = True Positives for class i
- `col.sum()` → full column = TP + FP

**Macro average** = calculate precision for each class, then take the mean (treats all classes equally)

In [44]:
def precesion(ytest, ypredtest):
    prec = []  # list to store precision for each class
    
    # Build confusion matrix
    cm = confusion_matrix(ytest, ypredtest)
    
    # Transpose so we can loop over columns as if they were rows
    # i = class index, col = that class's column
    for i, col in enumerate(cm.T):
        # col[i] = True Positives (diagonal)
        # col.sum() = TP + FP (entire column)
        preci = col[i] / col.sum()
        prec.append(preci)
    
    # Return macro average (equal weight to all classes)
    return np.mean(prec)

## Step 8 — Custom Recall Function

**Recall** = of all the actual instances of a class, how many did the model correctly identify?

$$Recall_i = \frac{TP}{TP + FN}$$

Recall is calculated **per row** of the confusion matrix:
- `row[i]` → diagonal = True Positives for class i
- `row.sum()` → full row = TP + FN

**Macro average** = mean of recall across all classes

In [58]:
def recall(ytest, ypredtest):
    recall = []  # list to store recall for each class
    
    # Build confusion matrix using the correct variables (ytest, not ytrain)
    cm = confusion_matrix(ytest, ypredtest)
    
    # Loop over rows — each row represents one actual class
    # i = class index, row = that class's row
    for i, row in enumerate(cm):
        # row[i] = True Positives (diagonal)
        # row.sum() = TP + FN (entire row)
        reci = row[i] / row.sum()
        recall.append(reci)
    
    # Return macro average (must use the list, not just the last value)
    return np.mean(recall)

## Step 9 — Verify Against Sklearn

We verify our custom functions match sklearn's built-in equivalents.

- `average='macro'` → tells sklearn to use macro averaging (same as our functions)
- All three comparisons should return `True`

**Why not use `np.isclose()` here?**  
Because our functions use the exact same arithmetic as sklearn — the values match exactly, not just approximately.

In [59]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

# Verify accuracy — should return True
accuracy(ytest, ypredtest) == accuracy_score(ytest, ypredtest)

In [60]:
# Verify precision — should return True
precesion(ytest, ypredtest) == precision_score(ytest, ypredtest, average='macro')

In [61]:
# Verify recall — should return True
recall(ytest, ypredtest) == recall_score(ytest, ypredtest, average='macro')

---
# Part 2: Linear Regression — California Housing Dataset

Here we switch from **classification** (predicting a category) to **regression** (predicting a continuous value).

We use the **California Housing dataset** to predict house prices (in $100,000 units).

The model learns the equation:
$$y = m_1x_1 + m_2x_2 + ... + m_8x_8 + c$$

Where each `x` is a feature (e.g. income, house age, rooms) and `m` is its learned weight.

> Note: We **overwrite** X and y here to point to the new dataset.

## Step 1 — Load the California Housing Dataset

The dataset has 8 features:
- MedInc, HouseAge, AveRooms, AveBedrms, Population, AveOccup, Latitude, Longitude

The target is the **median house value** in $100,000 units.

In [80]:
from sklearn.datasets import fetch_california_housing

# Load the dataset
data = fetch_california_housing()

# X = feature matrix (8 measurements per housing block)
X = data.data

# y = target (median house value in $100,000 units)
y = data.target

In [81]:
# Inspect the dataset structure — shows feature names, target names, and description
data

## Step 2 — Split into Train and Test Sets

Same 80/20 split as before. Always split **before** fitting.

In [82]:
from sklearn.model_selection import train_test_split

# 80% training, 20% test
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2)

## Step 3 — Fit the Linear Regression Model and Predict

- `.fit(Xtrain, ytrain)` → the model learns the slopes (`coef_`) and intercept
- `.predict(Xtest)` → applies the learned equation to unseen features

In [99]:
from sklearn.linear_model import LinearRegression

# Create and train the Linear Regression model
reg = LinearRegression()
reg.fit(Xtrain, ytrain)  # Learn from training data

# Predict house prices for the unseen test set
ypred = reg.predict(Xtest)

# Confirm the shapes of train/test splits
print(Xtrain.shape)  # (16512, 8) — 80% of data, 8 features
print(Xtest.shape)   # (4128, 8) — 20% of data, 8 features

## Step 4 — Inspect the Learned Coefficients

After fitting, the model stores:
- `reg.coef_` → the **slopes** (one per feature) — how much house price changes per unit of each feature
- `reg.intercept_` → the **intercept** (c in y = mx + c)

The full equation is: `y = coef_[0]*x1 + coef_[1]*x2 + ... + coef_[7]*x8 + intercept_`

Positive coefficient = price increases with that feature  
Negative coefficient = price decreases with that feature

In [100]:
# Print the 8 slope coefficients (one per feature)
print(reg.coef_)

In [101]:
# Print the intercept (baseline value when all features = 0)
print(reg.intercept_)

## Step 5 — Compare Predictions vs Actual Values

We look at the first 5 predictions vs actual values to get a quick sense of model accuracy.

- `ypred` = what the model **guessed**
- `ytest` = what the price **actually was**

In [102]:
# First 5 predicted prices (in $100,000 units)
print(ypred[:5])

In [103]:
# First 5 actual prices (in $100,000 units)
print(ytest[:5])

## Step 6 — Evaluate the Model

We use three standard regression metrics:

| Metric | Formula | Meaning |
|---|---|---|
| **MAE** | Mean of \|actual - predicted\| | Average error in same units as y |
| **MSE** | Mean of (actual - predicted)² | Penalises large errors more |
| **R²** | 1 - SS_res/SS_tot | How well the model explains the data (1 = perfect) |

**R² interpretation:**
- 0.9–1.0 = Excellent
- 0.7–0.9 = Good
- 0.5–0.7 = Moderate
- Below 0.5 = Poor

In [108]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Always pass actual values first, then predicted
print("MAE: ", mean_absolute_error(ytest, ypred))   # Average error in $100,000 units
print("MSE: ", mean_squared_error(ytest, ypred))    # Larger errors punished more
print("R2:  ", r2_score(ytest, ypred))              # How much variance is explained

## Step 7 — Visualise: Actual vs Predicted

This is the **standard way** to evaluate regression visually:

- Each **blue dot** = one house (actual price on x-axis, predicted price on y-axis)
- The **red line** = where dots would sit if predictions were **perfect** (y = x)
- Dots close to the red line = good predictions
- Dots far from the red line = poor predictions

> The spread of dots around the red line visually reflects the R² score.

In [117]:
# Scatter plot: actual vs predicted prices
plt.scatter(ytest, ypred, alpha=0.5)  # alpha makes overlapping dots more visible
plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")
plt.title("Actual vs Predicted")

# Red line = perfect prediction reference (y = x)
# NOT the regression line — just a visual benchmark
plt.plot([0, 5], [0, 5], 'r')

## Optional — Sorted Line Plot (Not Recommended)

This plot connects actual vs predicted values after sorting by actual price.
It becomes very jagged because many houses with the same actual price have very different predicted prices.

> The scatter plot above is the correct standard approach. This is shown here for learning purposes only.

In [120]:
# Sort test values by actual price so the x-axis is ordered
sorted_idx = np.argsort(ytest)

# Plot a line connecting actual vs predicted — becomes jagged with large datasets
plt.plot(ytest[sorted_idx], ypred[sorted_idx], 'r')